# Watsonx.ai Deployment Experiment

**Run comprehensive uncertainty quantification experiments and export for watsonx.ai**

This notebook:
1. Runs 42 experiments (7 epistemic × 6 aleatoric levels)
2. Analyzes all 7 attribution signals
3. Creates visualizations and comparisons
4. Exports best models for watsonx.ai deployment

**Runtime**: ~2-4 hours for all experiments (or run subset for testing)

**Output**: `/tmp/watsonx_sweep_experiments/sweep_TIMESTAMP/`

In [ ]:
# Imports
import json
import subprocess
import sys
from datetime import datetime
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
import yaml
from tqdm.notebook import tqdm

sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (14, 8)
# Standard library imports
import json
import subprocess
import sys
from datetime import datetime
from pathlib import Path
from typing import Dict, List, Tuple

# Third-party imports
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
import yaml
from tqdm.notebook import tqdm

# Set plotting style
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 8)
plt.rcParams['font.size'] = 10

print("✅ Imports successful")
print(f"Python version: {sys.version}")
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

In [ ]:
# Project paths
PROJECT_ROOT = Path.cwd()
SCRIPTS_DIR = PROJECT_ROOT / "scripts"
TRAINING_SCRIPT = SCRIPTS_DIR / "run_fast_uncertainty_classification.py"
CONFIG_TEMPLATE = PROJECT_ROOT / "experiments" / "configs" / "fast_uq_classification.yaml"

# Experiment output directory
EXPERIMENT_BASE_DIR = Path("/tmp/watsonx_sweep_experiments")
TIMESTAMP = datetime.now().strftime("%Y%m%d_%H%M%S")
EXPERIMENT_DIR = EXPERIMENT_BASE_DIR / f"sweep_{TIMESTAMP}"
EXPERIMENT_DIR.mkdir(parents=True, exist_ok=True)

# Results directories
RESULTS_DIR = EXPERIMENT_DIR / "results"
CONFIGS_DIR = EXPERIMENT_DIR / "configs"
ANALYSIS_DIR = EXPERIMENT_DIR / "analysis"
WATSONX_DIR = EXPERIMENT_DIR / "watsonx_packages"

for dir_path in [RESULTS_DIR, CONFIGS_DIR, ANALYSIS_DIR, WATSONX_DIR]:
    dir_path.mkdir(parents=True, exist_ok=True)

print(f"✅ Experiment directory: {EXPERIMENT_DIR}")
print(f"   - Results: {RESULTS_DIR}")
print(f"   - Configs: {CONFIGS_DIR}")
print(f"   - Analysis: {ANALYSIS_DIR}")
print(f"   - Watsonx packages: {WATSONX_DIR}")

In [ ]:
# Experiment parameters
EPISTEMIC_SWEEP = [1, 51, 101, 151, 201, 251, 301]  # under_train_per_class values
ALEATORIC_SWEEP = [0.0, 0.2, 0.4, 0.6, 0.8, 1.0]  # noise rates (simulated via eval_per_group)

# Fixed parameters (from config template)
FIXED_PARAMS = {
    "seed": 42,
    "device": "auto",
    "data": {
        "noise_type": "aggre_label",
        "under_supported_classes": "3,5",
        "regular_train_per_class": 300,
        "eval_per_group": 600,
    },
    "model": {
        "dinov2_model": "small",
        "hidden_dim": 256,
        "dropout": 0.2,
    },
    "training": {
        "epochs": 12,
        "learning_rate": 1e-3,
        "weight_decay": 1e-4,
        "train_batch_size": 256,
        "feature_batch_size": 64,
    },
    "evaluation": {
        "mc_passes": 20,
        "top_k": 10,
    },
    "paths": {
        "cifar10n_root": "./data/cifar10n",
        "feature_cache_dir": "./cache/fast_uncertainty_classification/features",
        "results_base_dir": str(RESULTS_DIR),
    },
}

print(f"✅ Parameter sweep configuration:")
print(f"   - Epistemic levels: {len(EPISTEMIC_SWEEP)} ({EPISTEMIC_SWEEP})")
print(f"   - Aleatoric levels: {len(ALEATORIC_SWEEP)} ({ALEATORIC_SWEEP})")
print(f"   - Total experiments: {len(EPISTEMIC_SWEEP) * len(ALEATORIC_SWEEP)}")
print(f"\n   Fixed parameters:")
print(f"   - Model: {FIXED_PARAMS['model']['dinov2_model']}")
print(f"   - Epochs: {FIXED_PARAMS['training']['epochs']}")
print(f"   - MC passes: {FIXED_PARAMS['evaluation']['mc_passes']}")

In [ ]:
# Signal names for analysis
SIGNAL_NAMES = [
    "msp_uncertainty",
    "predictive_entropy",
    "mutual_info",
    "inverse_coherence",
    "dominance",
    "inverse_mass",
    "inverse_logit_magnitude",
]

# Signal categories for visualization
SIGNAL_CATEGORIES = {
    "Baseline": ["msp_uncertainty", "predictive_entropy"],
    "Epistemic": ["mutual_info", "dominance", "inverse_mass"],
    "Aleatoric": ["inverse_coherence"],
    "Hybrid": ["inverse_logit_magnitude"],
}

print("✅ Signal configuration:")
for category, signals in SIGNAL_CATEGORIES.items():
    print(f"   - {category}: {', '.join(signals)}")